# Task EDA — Gemma 4 Developer Agent (Team B, Ishan)

CPU only (Colab or laptop). Input: `data/tasks.jsonl` (129 public tasks), downloaded with
`kaggle competitions download gemma-4-developer-agent -f tasks.jsonl -p data`.

**Which tasks:** design statistics are computed on the **train split** (`harness/splits/train.txt`, 99 tasks) only,
so nothing we decide is tuned on dev (rule 3). The repo counts at the top cover all 129 public tasks.
Hidden Kaggle tasks come from private repos, so these numbers describe the *kind* of task, not the exact repos.

In [1]:
import json, re
from collections import Counter
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "harness").is_dir():          # works from the repo root or team-b/notebooks/
    ROOT = ROOT.parent
FIG = ROOT / "team-b" / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

tasks = [json.loads(l) for l in open(ROOT / "data" / "tasks.jsonl", encoding="utf-8")]
split_of = {}
for s in ("train", "dev", "smoke"):
    for t in (ROOT / "harness" / "splits" / f"{s}.txt").read_text().split():
        split_of.setdefault(t, s if s != "smoke" else "train")   # smoke is a subset of train
print(len(tasks), "tasks;", Counter(split_of.get(t["instance_id"], "?") for t in tasks))

129 tasks; Counter({'train': 99, 'dev': 30})


## 1. Parse every reference patch

In [2]:
TEST_RE = re.compile(r"(^|/)(tests?|testing)/|(^|/)test_[^/]*\.py$|_test\.py$|conftest\.py$")
IDENT_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]{2,}")
DEF_RE = re.compile(r"^[+-]?\s*(?:async\s+)?(?:def|class)\s+([A-Za-z_][A-Za-z0-9_]*)")
HUNK_CTX_RE = re.compile(r"^@@[^@]*@@\s*(?:async\s+)?(?:def|class)\s+([A-Za-z_][A-Za-z0-9_]*)")

def parse_patch(patch: str):
    files, cur = {}, None
    for line in patch.splitlines():
        if line.startswith("+++ "):
            path = line[4:].strip()
            path = path[2:] if path.startswith("b/") else path
            cur = files.setdefault(path, {"added": 0, "removed": 0, "hunks": 0, "new_file": False, "symbols": set()})
        elif line.startswith("--- "):
            if line.strip() == "--- /dev/null":
                pending_new = True
            continue
        elif cur is None:
            continue
        elif line.startswith("@@"):
            cur["hunks"] += 1
            if line.startswith("@@ -0,0 "):
                cur["new_file"] = True
            m = HUNK_CTX_RE.match(line)
            if m:
                cur["symbols"].add(m.group(1))
        elif line.startswith("+"):
            cur["added"] += 1
            m = DEF_RE.match(line)
            if m: cur["symbols"].add(m.group(1))
        elif line.startswith("-"):
            cur["removed"] += 1
            m = DEF_RE.match(line)
            if m: cur["symbols"].add(m.group(1))
    return files

rows = []
for t in tasks:
    files = parse_patch(t["patch"])
    src = {p: f for p, f in files.items() if not TEST_RE.search(p)}
    issue = t["problem_statement"] or ""
    issue_l = issue.lower()
    idents = set(IDENT_RE.findall(issue))
    syms = set().union(*[f["symbols"] for f in src.values()]) if src else set()
    def named(p):
        stem = Path(p).stem
        dotted = p[:-3].replace("/", ".") if p.endswith(".py") else p
        return (p.lower() in issue_l or Path(p).name.lower() in issue_l
                or dotted.lower() in issue_l or (len(stem) > 3 and stem in idents and stem not in {"utils", "main", "base", "types"}))
    rows.append({
        "instance_id": t["instance_id"],
        "repo": t["repo"].split("/")[-1],
        "split": split_of.get(t["instance_id"], "?"),
        "issue_chars": len(issue),
        "issue_words": len(issue.split()),
        "has_traceback": "Traceback (most recent call last)" in issue,
        "has_code_block": "```" in issue,
        "n_src_files": len(src),
        "n_test_files_in_patch": len(files) - len(src),
        "lines_changed": sum(f["added"] + f["removed"] for f in src.values()),
        "hunks": sum(f["hunks"] for f in src.values()),
        "adds_new_file": any(f["new_file"] for f in src.values()),
        "file_named_in_issue": any(named(p) for p in src),
        "symbol_named_in_issue": bool(syms & idents),
        "n_symbols_touched": len(syms),
        "n_test_files_test_patch": len(parse_patch(t["test_patch"] or "")),
    })
df = pd.DataFrame(rows)
train = df[df.split == "train"].copy()
df.head()

,instance_id,repo,split,issue_chars,issue_words,has_traceback,has_code_block,n_src_files,n_test_files_in_patch,lines_changed,hunks,adds_new_file,file_named_in_issue,symbol_named_in_issue,n_symbols_touched,n_test_files_test_patch
0,fastapi_15661,fastapi,train,1025,172,False,False,1,0,216,1,True,False,False,9,1
1,fastapi_15588,fastapi,dev,1891,323,False,False,1,0,23,2,False,False,False,6,1
2,fastapi_15589,fastapi,train,1247,206,False,False,1,0,4,1,False,False,False,1,1
3,fastapi_15030,fastapi,train,148,26,False,False,9,0,560,17,True,False,False,25,6
4,fastapi_14962,fastapi,train,210,34,False,False,3,0,55,6,False,False,False,4,3


## 2. Tasks per repo (all 129 public tasks)

In [3]:
per_repo = pd.crosstab(df.repo, df.split)
per_repo["total"] = per_repo.sum(axis=1)
print(per_repo)
fig, ax = plt.subplots(figsize=(6, 3))
per_repo.drop(columns="total").plot.barh(ax=ax, stacked=True, color=["#2f6fdf", "#9aa5b1"])
ax.set_xlabel("tasks"); ax.set_ylabel(""); ax.set_title("Public tasks per repo and split")
plt.tight_layout(); plt.savefig(FIG / "eda_tasks_per_repo.png", dpi=150); plt.show(); plt.close("all")

split     dev  train  total
repo                       
fastapi    15     52     67
httpx       1      0      1
requests    3     10     13
rich       11     37     48


C:\Users\Ishan M Shah\AppData\Local\Temp\ipykernel_35104\3024890167.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "eda_tasks_per_repo.png", dpi=150); plt.show(); plt.close("all")


## 3. How big are the fixes? (train)

In [4]:
summary = {
    "train tasks": len(train),
    "single-source-file fixes": f"{(train.n_src_files == 1).mean():.0%}",
    "<= 3 source files": f"{(train.n_src_files <= 3).mean():.0%}",
    "median source lines changed": int(train.lines_changed.median()),
    "fixes <= 20 lines": f"{(train.lines_changed <= 20).mean():.0%}",
    "median hunks": int(train.hunks.median()),
    "patch adds a new source file": f"{train.adds_new_file.mean():.0%}",
    "patches with no source file": int((train.n_src_files == 0).sum()),
}
for k, v in summary.items(): print(f"{k:32s} {v}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
train.n_src_files.clip(upper=6).value_counts().sort_index().plot.bar(ax=axes[0], color="#2f6fdf")
axes[0].set_title("Source files changed per fix (train)"); axes[0].set_xlabel("files (6 = 6+)"); axes[0].set_ylabel("tasks")
axes[0].tick_params(axis="x", rotation=0)
train.lines_changed.clip(upper=200).plot.hist(ax=axes[1], bins=20, color="#2f6fdf")
axes[1].set_title("Source lines changed per fix (train)"); axes[1].set_xlabel("lines added + removed (200 = 200+)")
plt.tight_layout(); plt.savefig(FIG / "eda_fix_size.png", dpi=150); plt.show(); plt.close("all")

train tasks                      99
single-source-file fixes         71%
<= 3 source files                91%
median source lines changed      12
fixes <= 20 lines                64%
median hunks                     2
patch adds a new source file     8%
patches with no source file      0


C:\Users\Ishan M Shah\AppData\Local\Temp\ipykernel_35104\1307994356.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "eda_fix_size.png", dpi=150); plt.show(); plt.close("all")


## 4. Does the issue tell us where to look? (train)

In [5]:
loc = pd.Series({
    "changed file named in issue": train.file_named_in_issue.mean(),
    "changed function/class named in issue": train.symbol_named_in_issue.mean(),
    "either": (train.file_named_in_issue | train.symbol_named_in_issue).mean(),
    "issue has a traceback": train.has_traceback.mean(),
    "issue has a code block": train.has_code_block.mean(),
})
print((loc * 100).round(0).astype(int).astype(str) + "%")
by_repo = train.groupby("repo")[["file_named_in_issue", "symbol_named_in_issue"]].mean().mul(100).round(0)
print(by_repo)
fig, ax = plt.subplots(figsize=(6, 3))
(loc * 100).plot.barh(ax=ax, color="#2f6fdf")
ax.set_xlim(0, 100); ax.set_xlabel("% of train tasks"); ax.set_title("Localization signals in the issue text")
plt.tight_layout(); plt.savefig(FIG / "eda_localization_signals.png", dpi=150); plt.show(); plt.close("all")

changed file named in issue              21%
changed function/class named in issue    23%
either                                   37%
issue has a traceback                     0%
issue has a code block                   10%
dtype: object
          file_named_in_issue  symbol_named_in_issue
repo                                                
fastapi                  15.0                   23.0
requests                 10.0                   40.0
rich                     32.0                   19.0


C:\Users\Ishan M Shah\AppData\Local\Temp\ipykernel_35104\3542130132.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "eda_localization_signals.png", dpi=150); plt.show(); plt.close("all")


## 5. Issue length (train)

In [6]:
print(train.issue_words.describe().round(0))
fig, ax = plt.subplots(figsize=(6, 3))
train.issue_words.clip(upper=1500).plot.hist(ax=ax, bins=25, color="#2f6fdf")
ax.set_xlabel("words in problem statement (1500 = 1500+)"); ax.set_title("Issue length (train)")
plt.tight_layout(); plt.savefig(FIG / "eda_issue_length.png", dpi=150); plt.show(); plt.close("all")

count     99.0
mean      98.0
std      138.0
min        4.0
25%       14.0
50%       48.0
75%      118.0
max      772.0
Name: issue_words, dtype: float64


C:\Users\Ishan M Shah\AppData\Local\Temp\ipykernel_35104\1359754767.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "eda_issue_length.png", dpi=150); plt.show(); plt.close("all")


## 5b. What kind of text is the problem statement? (train)

In [7]:
ps = {t["instance_id"]: t["problem_statement"] or "" for t in tasks}
train["short_lt_20_words"] = train.issue_words < 20
train["only_links_issue"] = train.instance_id.map(lambda i: bool(re.search(r"(issues/\d+|#\d+)", ps[i]))) & (train.issue_words < 40)
train["pr_template"] = train.instance_id.map(lambda i: "## Pull Request" in ps[i])
kind = train[["short_lt_20_words", "only_links_issue", "pr_template", "has_code_block"]].mean().mul(100).round(0)
print(kind.astype(int).astype(str) + "%")
for i in train.instance_id.sample(4, random_state=266):
    print("-----", i); print(ps[i][:250])

short_lt_20_words    27%
only_links_issue     29%
pr_template           6%
has_code_block       10%
dtype: object
----- requests_6644
Trim excess leading path separators

A URL with excess leading / (path-separator)s would cause urllib3 to attempt to reparse the request-uri as a full URI with a host and port. This bypasses that logic in ConnectionPool.urlopen by replacing these lea
----- rich_4079
Inline table code

Fixes https://github.com/Textualize/rich/issues/4038
----- fastapi_14512
🐛 Fix support for tagged union with discriminator inside of `Annotated` with `Body()`

🐛 Fix support for tagged union with discriminator inside of `Annotated` with `Body()`

Related to: https://github.com/fastapi/fastapi/discussions/14495, https://
----- fastapi_14962
✨ Serialize JSON response with Pydantic (in Rust), when there's a Pydantic return type or response model

✨ Serialize JSON response with Pydantic (in Rust), when there's a Pydantic return type or response model


## 6. Save the per-task table

In [8]:
df.to_csv(ROOT / "team-b" / "docs" / "task_eda_table.csv", index=False)
summary, loc.round(2).to_dict()

({'train tasks': 99,
  'single-source-file fixes': '71%',
  '<= 3 source files': '91%',
  'median source lines changed': 12,
  'fixes <= 20 lines': '64%',
  'median hunks': 2,
  'patch adds a new source file': '8%',
  'patches with no source file': 0},
 {'changed file named in issue': 0.21,
  'changed function/class named in issue': 0.23,
  'either': 0.37,
  'issue has a traceback': 0.0,
  'issue has a code block': 0.1})